In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import numpy.fft as fft
import scipy.signal
import tqdm

import utils
from utils import collect_metadata_utils, sample_streaming

utils.plotting.setup_default_plotting()

In [ ]:
# Runtime config (COLLECTS_PATH etc.) comes from `.env` -- see utils/environment_variables.py
# and the README for the template.
collects_dir = utils.environment_variables.get_collects_path()
experiment_names = collect_metadata_utils.list_experiment_names(collects_dir)
print("Available experiments:", ", ".join(experiment_names))
experiment_name = experiment_names[3]  # change index to select a different experiment

# Note: I keep a metadata.yml file in each experiment directory to keep track of sample metadata
#  and what are the available collect filenames.  You can either create your own metadata.yml,
#  or modify the code below to directly chose your data filepath and set sample parameters.
metadata_filepath = collects_dir / experiment_name / "metadata.yml"
metadata = collect_metadata_utils.load_experiment_metadata_from_file(metadata_filepath, print_summary=True)

collect_id = metadata.collect_ids[0]  # change index to select a different collect (see printed list above)
band_id = metadata.band_ids[0]  # change index to select a different band

resolved = collect_metadata_utils.resolve_collect(collects_dir, experiment_name, collect_id, band_id)
collect_filepath = resolved.collect_filepath
samp_rate = resolved.samp_rate
sample_params = resolved.sample_params
inter_freq_l1_hz = resolved.inter_freq_hz

In [ ]:
fig = plt.figure(figsize=(12, 2), dpi=150)
utils.plotting.plot_receiver_channel_bands(fig, metadata)

In [ ]:
buffer_duration_ms = 40
buffer_size_samples = int(samp_rate * buffer_duration_ms / 1e3)
buffer_size_bytes = sample_streaming.compute_sample_array_size_bytes(
    buffer_size_samples, sample_params.bit_depth, sample_params.is_complex
)
byte_buffer = bytearray(buffer_size_bytes)
sample_buffer = np.zeros(buffer_size_samples, dtype=np.complex64)

with open(collect_filepath, "rb") as f:
    f.readinto(byte_buffer)

sample_streaming.convert_to_complex64_samples(
    byte_buffer,
    sample_buffer,
    sample_params
)
baseband_sample_buffer = np.zeros(buffer_size_samples, dtype=np.complex64)
sample_streaming.mixdown_samples(
    sample_buffer,
    baseband_sample_buffer,
    samp_rate,
    0.0,
    inter_freq_l1_hz
)

baseband_sample_buffer -= np.mean(baseband_sample_buffer)
# baseband_sample_buffer *= np.exp(-1j * np.angle(np.mean(np.exp(1j * np.angle(baseband_sample_buffer)))))

In [ ]:
fig = plt.figure(figsize=(10, 4))
axes = fig.subplots(1, 2, width_ratios=[1.5, 1])
ax1: plt.Axes = axes[0]
ax2: plt.Axes = axes[1]
hist_bins = np.arange(-2**(sample_params.bit_depth-1), 2**(sample_params.bit_depth-1))
ax1.hist(baseband_sample_buffer.real, bins=hist_bins, histtype="stepfilled", color="r", alpha=0.6, align="left", label="Real")
ax1.hist(baseband_sample_buffer.imag, bins=hist_bins, histtype="stepfilled", color="b", alpha=0.6, align="mid", label="Imaginary")
ax1.set_xlabel("Sample Value")
ax1.set_ylabel("Count")
ax1.grid()
ax1.legend(loc="upper right")

ax2.scatter(baseband_sample_buffer.real, baseband_sample_buffer.imag, color="k", s=1, alpha=0.01, zorder=1)
ax2.set_axisbelow(True)
ax2.grid()
ax2.set_xlabel("Real")
ax2.set_ylabel("Imaginary")

plt.tight_layout()
plt.show()

# NOTE: should see normal-looking distribution for raw samples.  But if samples are real, the imaginary component should be all zeros.

In [ ]:
# Plot Welch PSD estimate of the samples
fig = plt.figure(figsize=(10, 4))
utils.plotting.plot_welch_psd(fig, sample_buffer, baseband_sample_buffer, samp_rate)
plt.show()

In [ ]:
stft_interval_ms = 1000
buffer_skip = stft_interval_ms // buffer_duration_ms
periodogram_nperseg = 4096
periodogram_noverlap = 2048
stft_freq = fft.fftshift(fft.fftfreq(periodogram_nperseg, 1 / samp_rate))
# stft = ShortTimeFFT(hamming(buffer_size_samples), buffer_size_samples, samp_rate)

file_duration_estimate_ms = os.path.getsize(collect_filepath) // buffer_size_bytes * buffer_duration_ms
num_windows = file_duration_estimate_ms // stft_interval_ms
periodogram = np.zeros((num_windows, periodogram_nperseg))

# compute STFT periodogram of the signal
with sample_streaming.FileSampleStream(
        collect_filepath,
        sample_params,
        buffer_size_samples,
    ) as sample_stream:

    sample_buffer_generator = sample_stream.sample_buffer_generator(skip=buffer_skip)
    for i, sample_buffer in enumerate(tqdm.tqdm(sample_buffer_generator, total=num_windows)):
        _, epoch_psd = scipy.signal.welch(
            sample_buffer,
            fs=samp_rate,
            nperseg=periodogram_nperseg,
            noverlap=periodogram_noverlap,
            window="hann",
            return_onesided=False,
            scaling="density",
        )
        periodogram[i, :] = fft.fftshift(epoch_psd)
        
        if i + 1 == num_windows:
            break

In [ ]:
# Plot the periodogram
fig = plt.figure(figsize=(10, 4))
plt.imshow(10 * np.log10(periodogram.T), aspect="auto", origin="lower", extent=[0, num_windows * stft_interval_ms / 1000, stft_freq[0]/1e6, stft_freq[-1]/1e6], interpolation="nearest")
plt.xlabel("Time (s)")
plt.ylabel("Frequency (MHz)")
plt.title("STFT Periodogram")
plt.colorbar(label="Power/Frequency (dB/Hz)")
plt.show()